import libraries

In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv("/content/Tweets.csv")
df.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

select columns

In [ ]:
df= df[["airline_sentiment", "text"]]
df.columns = ["sentiment", "text"]
df.head()

In [ ]:
def clean_text(text):
  text = text.lower()
  text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
  text = re.sub(r'\@\w+|\#', '', text)
  text = re.sub(r"[^a-zA-Z\s]", '', text)
  return text

df["clean_text"] = df["text"].apply(clean_text)
df.head()

label

In [ ]:
label_map = {"negative":0, "positive":1}

df = df[df["sentiment"].isin(label_map.keys())]

df["label"] = df["sentiment"].map(label_map)

df["label"].value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.2,
    random_state=42
)

TF-IDF

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

train model

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

evaluate

In [ ]:
predictions = model.predict(X_test_vec)
print(accuracy_score(y_test, predictions))




SECTION2: BERT MODEL
**bold text**

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset

split again

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.2,
    random_state=42
)

Tokenizer

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
train_encodings = tokenizer(train_texts.tolist(), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts.tolist(), truncation=True, padding=True, max_length=128)

dataset

In [ ]:
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": train_labels.tolist()
})

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": test_labels.tolist()
})

Model

In [ ]:
bert_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Training Setup

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    learning_rate=2e-5
)

Metrics

In [ ]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

Train, most time consuming along with TPU

In [ ]:
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

## Evaluate
eval_accu=0.94

In [ ]:
trainer.evaluate()

# SECTION3:**RoBERT**

Load RoBERT


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

Tokenize

In [ ]:
train_encodings = roberta_tokenizer(train_texts.tolist(), truncation=True, padding=True, max_length=128)
test_encodings = roberta_tokenizer(test_texts.tolist(), truncation=True, padding=True, max_length=128)

Dataset

In [ ]:
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": train_labels.tolist()
})

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": test_labels.tolist()
})

Train RoBERT

In [ ]:
trainer = Trainer(
    model=roberta_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

**Confusion Matrix**

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

predictions = trainer.predict(test_dataset)

import numpy as np
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = test_labels.values

In [ ]:
cm = confusion_matrix(y_true, y_pred)

sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
print(classification_report(y_true, y_pred))

In [ ]:
!pip install wordcloud

In [ ]:
from wordcloud import WordCloud

text = " ".join(df["clean_text"])

wc = WordCloud(width=800, height=400).generate(text)

plt.imshow(wc)
plt.axis("off")
plt.show()